# Faster R-CNN — Forge Sweeps (Kaggle T4 × 2)

Sẵn cấu hình **preflight: sandbox / seed42 / 2 epoch / 1 trial / shared test policy**.
Run All hoặc Save Version → Save & Run All sẽ đăng ký Sweep và bắt đầu dùng GPU quota.
Không phải notebook chỉ xem config. Nếu muốn dừng trước training, không chạy cell agent.

Notebook riêng, không thay notebook03 của nhóm. Thứ tự: preflight → pilot → tuning.
Một agent chạy tuần tự; mỗi trial dùng hai GPU. Objective là best validation mAP50–95.
Preflight/pilot/tuning đều vào sandbox; chỉ run final sau khi khóa protocol mới dùng BTL.

Kaggle: Add Input cache đã tiền xử lý, GPU T4 × 2, Internet On,
Settings → Secrets → bật WANDB_API_KEY hiện có. Không ghi hoặc in key.
Code pipeline pin commit 9ec948e đã kiểm. Khi đổi source có chủ ý, cập nhật REPO_REF bằng SHA đã push.
Kết thúc có đúng một bundle ZIP cùng SHA256 ngay dưới /kaggle/working để tải từ Output.

In [ ]:
import os, sys, subprocess, json, re
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Detection.git"
REPO_REF = "9ec948ebc41aaca7e2c7c64b6bb4d95e2ea81266"  # SHA đầy đủ của commit có tích hợp Sweeps, đã push; không dùng tên branch.
assert re.fullmatch(r"[0-9a-f]{40}", REPO_REF), "Điền SHA đã commit/push trước khi chạy."
assert Path("/kaggle/working").exists(), "Notebook này dành cho Kaggle."
REPO = Path("/kaggle/working/VN-Traffic-Sign-Detection")

def run(command, **kwargs):
    return subprocess.run([str(x) for x in command], check=True, **kwargs)

if not (REPO / ".git").exists():
    run(["git", "clone", "--no-checkout", REPO_URL, REPO])
else:
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty.strip(), "Repo có tracked changes; giữ lại trước khi checkout."
run(["git", "-C", REPO, "fetch", "origin", REPO_REF])
run(["git", "-C", REPO, "checkout", "--detach", REPO_REF])
os.chdir(REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[detect]", "wandb==0.31.0"])
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 1. Kiểm GPU và secret

Cell này chưa tạo run hoặc Sweep. API host lấy từ config chung của nhóm.
Lỗi CUDA/NMS hoặc thiếu GPU: dừng, không tự giảm batch hay số GPU.

In [ ]:
import torch, torchvision, wandb
from torchvision.ops import nms
from kaggle_secrets import UserSecretsClient
from vn_tsd.utils.io import load_yaml, save_yaml

shared = load_yaml(REPO / "configs/shared.yaml")
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_BASE_URL"] = shared["project"]["wandb_base_url"]
os.environ["WANDB_MODE"] = "online"
print("torch", torch.__version__, "torchvision", torchvision.__version__, "wandb", wandb.__version__)
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Cần đúng hai GPU T4."
for index in range(2):
    device = f"cuda:{index}"
    result = nms(torch.tensor([[0., 0., 10., 10.]], device=device), torch.tensor([.9], device=device), .5)
    assert result.tolist() == [0]
    print(index, torch.cuda.get_device_name(index))

## 2. Chọn giai đoạn và chuẩn bị offline

**preflight:** một trial cố định, 2 epoch; chỉ kiểm kỹ thuật, không xếp hạng với full train.
**pilot:** một trial baseline cố định, ngân sách epoch hiện hành trong YAML (hiện 100), patience hiện hành.
Đọc best/last epoch, LR, early stopping, thời gian, VRAM rồi chốt budget trước tuning.
**tuning:** search space LR/WD/dropout/jitter; phải điền `TUNING_RUN_CAP` và `TUNING_EPOCHS` sau pilot.

`TEST_POLICY="shared"` giữ policy auto-test của config nhóm. Nếu nhóm đồng ý không đánh giá test
trong tuning, đổi thành `"skip"` cho một campaign mới. Không dùng test để chọn trial.
Chuẩn bị sẽ quét/hash toàn cache, có thể mất vài phút; không copy dataset vào Working.

In [ ]:
from vn_tsd.config.resolve import resolve_config
from vn_tsd.pipelines.faster_rcnn.sweep import prepare, load_campaign, PARAMETERS, OBJECTIVE

STAGE = "preflight"  # preflight / pilot / tuning
CAMPAIGN_NAME = "frcnn_preflight_s42_sandbox_v1"  # tên mới mỗi campaign; giữ tên khi tiếp tục cùng campaign
CAMPAIGN = Path("/kaggle/working/frcnn_sweeps") / CAMPAIGN_NAME
DATA_ROOT = Path(load_yaml("configs/runtime/kaggle.yaml")["data_root_default"])
WANDB_PROJECT = "sandbox"  # preflight/pilot/tuning; BTL dành cho final đã chốt
TEST_POLICY = "shared"
TUNING_RUN_CAP = None  # chốt sau pilot, ví dụ hữu hạn; không tự dùng 12 như budget đã duyệt
TUNING_EPOCHS = None   # chốt sau pilot; đổi cap làm đổi cosine từ đầu
assert STAGE in {"preflight", "pilot", "tuning"}
assert DATA_ROOT.is_dir(), f"Kiểm lại Add Input/cache path: {DATA_ROOT}"

overrides = {"data": {"root": str(DATA_ROOT)}, "seed": 42,
             "project": {"wandb_project": WANDB_PROJECT}}
if STAGE == "preflight":
    overrides["train"] = {"epochs": 2}
elif STAGE == "tuning":
    assert type(TUNING_RUN_CAP) is int and TUNING_RUN_CAP > 0
    assert type(TUNING_EPOCHS) is int and TUNING_EPOCHS > 0
    overrides["train"] = {"epochs": TUNING_EPOCHS}
cfg = resolve_config("configs/pipelines/faster_rcnn.yaml", "configs/shared.yaml",
                     "configs/runtime/kaggle.yaml", overrides=overrides)
search = load_yaml("configs/sweeps/faster_rcnn.yaml")
cap = TUNING_RUN_CAP if STAGE == "tuning" else 1
if STAGE != "tuning":
    search = {"method": "grid", "metric": {"name": OBJECTIVE, "goal": "maximize"},
              "parameters": {key: {"value": cfg[key.split('.')[0]][key.split('.')[1]]} for key in PARAMETERS}}
if CAMPAIGN.exists():
    frozen, existing, existing_search = load_campaign(CAMPAIGN)
    # Existing campaigns are immutable. Avoid silently resuming the wrong stage/settings.
    from vn_tsd.pipelines.faster_rcnn.dataset import content_hash
    requested = json.loads((CAMPAIGN / "notebook_request.json").read_text())
    assert requested == {"stage": STAGE, "cap": cap, "test_policy": TEST_POLICY, "cfg": cfg, "search": search}
    print("Giữ campaign đã chuẩn bị:", CAMPAIGN)
else:
    prepare(CAMPAIGN, cfg, search, run_cap=cap, nproc=2, test_policy=TEST_POLICY)
    (CAMPAIGN / "notebook_request.json").write_text(json.dumps(
        {"stage": STAGE, "cap": cap, "test_policy": TEST_POLICY, "cfg": cfg, "search": search}, indent=2))
print((CAMPAIGN / "base_config.yaml").read_text())
print((CAMPAIGN / "sweep.yaml").read_text())

## 3. Đăng ký Sweep — kết nối Forge sandbox

Đã bật REGISTER_SWEEP=True. Chạy cell sẽ tạo Sweep trong sandbox theo campaign config,
hoặc dùng ID đã lưu khi chạy lại. Chưa bắt đầu training ở cell này.
Nếu có registration_started.json nhưng thiếu sweep_id.json, kiểm dashboard/khôi phục ID thật;
không xóa marker rồi đăng ký lại mù vì có thể tạo trùng Sweep.

In [ ]:
REGISTER_SWEEP = True
if REGISTER_SWEEP:
    run([sys.executable, "-m", "vn_tsd.pipelines.faster_rcnn.sweep", "register", "--campaign", CAMPAIGN])
    print((CAMPAIGN / "sweep_id.json").read_text())
else:
    print("Chưa đăng ký. Bật REGISTER_SWEEP sau khi review config.")

## 4. Chạy một trial preflight — bắt đầu dùng GPU quota

Đã bật START_AGENT=True và AGENT_COUNT=1. Run All/Save & Run All thực sự train2epochs trên2GPU.
Không chạy hai agent cùng GPU. Pilot/tuning tiếp theo giữ sandbox, đổi stage và campaign name;
tuning chỉ bắt đầu sau khi chốt TUNING_RUN_CAP/TUNING_EPOCHS qua pilot.

Pause/Stop Sweep để ngừng nhận trial mới; Cancel/stop notebook có thể ngắt trial hiện tại.
Giữ logs trial dở; checkpoint v1 không resume training, retry cần runID mới.
Nếu cellagent lỗi, vẫn chạy cellexport riêng bên dưới để giữ artifacts điều tra.
Không chạy lại training chỉ để tải bundle.

In [ ]:
START_AGENT = True
AGENT_COUNT = 1
if START_AGENT:
    assert (CAMPAIGN / "sweep_id.json").is_file(), "Đăng ký/khôi phục sweep ID trước."
    run([sys.executable, "-m", "vn_tsd.pipelines.faster_rcnn.sweep", "agent",
         "--campaign", CAMPAIGN, "--count", AGENT_COUNT])
else:
    print("Chưa chạy GPU. Bật START_AGENT khi sẵn sàng.")

## 5. Tải kết quả — một ZIP và checksum ở Output gốc

Cell này chỉ export, không đăng ký Sweep/train. Có thể chạy riêng sau cellagent lỗi hoặc training xong.
Bundle chứa metadata campaign và **mọi trial**: config, metrics/history, checkpoint, logs, figures;
không lồng run.zip hoặc checksum cũ để tránh nhân đôi checkpoint.
ZIP và .sha256 nằm trực tiếp trong /kaggle/working, tương tự notebook fulltrain trước đây.
Giữ cả failed/interrupted. SaveVersion giữ outputs; tải hai file đưa vào
`.agents/artifacts/notebooks/<campaign>/`. Forge hiện log metrics/figures, chưa tự upload checkpoint Artifact.

In [ ]:
import hashlib, uuid
from zipfile import ZipFile, ZIP_STORED
from IPython.display import display, FileLink

# Có thể copy nguyên cell này sang session cũ: CAMPAIGN phải trỏ campaign đã chạy.
assert CAMPAIGN.is_dir(), f"Chưa có campaign: {CAMPAIGN}"
bundle = Path("/kaggle/working") / f"{CAMPAIGN.name}-artifacts-{uuid.uuid4().hex[:8]}.zip"
files = sorted(path for path in CAMPAIGN.rglob("*")
               if path.is_file() and path.suffix.lower() not in {".zip", ".sha256"}
               and path.name not in {".netrc", "_netrc", ".env"})
assert files, "Campaign chưa có file để export"
with ZipFile(bundle, "x", compression=ZIP_STORED, allowZip64=True) as archive:
    for path in files:
        archive.write(path, f"{CAMPAIGN.name}/{path.relative_to(CAMPAIGN).as_posix()}")
with ZipFile(bundle) as archive:
    assert archive.testzip() is None, "Bundle CRC error"
digest = hashlib.sha256()
with bundle.open("rb") as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        digest.update(block)
checksum = bundle.with_suffix(bundle.suffix + ".sha256")
checksum.write_text(f"{digest.hexdigest()}  {bundle.name}\n", encoding="utf-8")
print("Tải hai file ở Output /kaggle/working:")
print(bundle)
print(checksum)
display(FileLink(str(bundle)))
display(FileLink(str(checksum)))
for trial_dir in sorted((CAMPAIGN / "trials").glob("*")):
    status = trial_dir / "launch_status.json"
    print(trial_dir.name, json.loads(status.read_text()) if status.exists() else "chưa có launch receipt")

Nguồn SDK/command macros: [CoreWeave Sweeps walkthrough](https://docs.coreweave.com/products/wandb/sweeps/walkthrough),
[Sweep configuration](https://docs.coreweave.com/products/wandb/sweeps/sweep-config-keys).
Local mock/CPU-Gloo tests không thay nghiệm thu Kaggle CUDA/NCCL/Forge thật.